# Getting Started with NePS: Basic HPO
This tutorial introduces **Hyperparameter Optimization (HPO)** with NePS, starting with synthetic functions and progressing to real deep learning tasks.

## Installation
Requires Python 3.10+. Install NePS via:

In [2]:
# %%
!pip install neural-pipeline-search

/home/alipourn/neps/.venv/bin/python


## Example 1: Synthetic Function Optimization
We start with a simple optimization problem to understand NePS basics.

In [3]:
# %%
import math
import neps
import logging

ModuleNotFoundError: No module named 'neps'

In [ ]:
logging.basicConfig(level=logging.INFO)

In [ ]:
def branin(x1, x2):
    """Minimize the 2D Branin function.
    
    Reference: https://www.sfu.ca/~ssurjano/branin.html
    """
    a = 1
    b = 5.1 / (4 * math.pi**2)
    c = 5 / math.pi
    r = 6
    s = 10
    t = 1 / 8 * math.pi

    f = a * (x2 - b * x1**2 + c * x1 - r) ** 2 + s * (1 - t) * math.cos(x1) + s
    return f

In [ ]:
# %%
# Define the search space and run optimization
class BraninSpace(neps.PipelineSpace):
    x1 = neps.Float(-5, 10)
    x2 = neps.Float(0, 15)

In [ ]:
neps.run(
    pipeline_space=BraninSpace(),
    root_directory="branin_demo/",
    evaluations_to_spend=25,
    evaluate_pipeline=branin,
    optimizer="random_search",
)

In [ ]:
# Check the optimization results:
!tail ./branin_demo/summary/best_config_trajectory.txt

Great! NePS found the minimum loss over 25 evaluations. 
The NePS workflow always follows this pattern:
1. **Define** an objective function
2. **Specify** a search space
3. **Call** `neps.run()` to optimize

## Example 2: Deep Learning HPO
Now we optimize hyperparameters for a neural network on MNIST (multi-class classification with CNNs).

For the full training code, see [train.py](https://github.com/automl/neps/blob/master/tutorials/train.py).

In [ ]:
# %%
from tutorials.train import training_pipeline
from tutorials.utils import set_seeds

In [ ]:
# Test the training pipeline with a small subset
training_pipeline(
    # checkpoint controller
    out_dir=None,
    load_dir=None,
    # possible hyperparameters
    num_layers=1,
    batch_size=512,
    # possible fidelities
    subsample=0.1,
    epochs=2,
    val_fraction=0.3,
    allow_checkpointing=False,
    verbose=True,
)

Note: `objective_to_minimize = 1 - val_accuracy` (validation error), since NePS minimizes.

### Running the HPO
Now we apply the standard NePS pattern to optimize the training pipeline.

In [ ]:
# Step 1: Create a NePS wrapper
def evaluate_pipeline(
    pipeline_directory,  # For saving checkpoints
    previous_pipeline_directory,  # For loading checkpoints
    **hyperparameters,  # Determined by HPO algorithms and passed by NePS
):
    return training_pipeline(
        **hyperparameters,
        out_dir=pipeline_directory,
        load_dir=previous_pipeline_directory,
        # misc settings
        batch_size=512,
        log_neps_tensorboard=True,
        verbose=False,
        allow_checkpointing=True,
        use_for_demo=True  # toggle as desired
    )

In [ ]:
# Step 2: Define the search space (tuning learning rate)
pipeline_space = dict(
    learning_rate=neps.Float(1e-6, 1e-1, log=True),
)

In [ ]:
# Step 3: Run optimization
logging.basicConfig(level=logging.INFO, force=True)
set_seeds(1)
neps.run(
    evaluate_pipeline=evaluate_pipeline,
    root_directory="results_hpo_demo",
    pipeline_space=pipeline_space,
    evaluations_to_spend=3  # HPO budget
)

### Analyzing Results
Check the status and outputs:

In [ ]:
!python -m neps.status results_hpo_demo/

In [ ]:
# %%
# View the summary files:
!cat results_hpo_demo/summary/short.csv

In [ ]:
!cat results_hpo_demo/summary/best_config_trajectory.txt

In [ ]:
!cat results_hpo_demo/summary/best_config.txt

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("results_hpo_demo/summary/full.csv")
df.head()

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
# Load the tensorboard
%load_ext tensorboard

In [ ]:
%tensorboard --logdir /content/neps_colab_playground/results_hpo_demo

In [ ]:
"""For more details on the Tensorboard integration, see documentation [here](https://automl.github.io/neps/latest/reference/analyse/#visualizing-results).

## A slightly more elaborate HPO

Now, we construct a slightly more elaborate search space and perform a longer HPO run.

<!-- *NOTE*: If using a GPU or with time in hand, we recommend `run_pipeline()` or else `run_pipeline_demo()` for shorter, quicker runs. -->

<!-- *NOTE, again*: Using `run_pipeline_demo()` may affect the multi-fidelity runs in this notebook. -->
"""

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
# (Optional) Activate live view of Tensorboard output
# NOTE: this will populate once the next run is active and `results_hpo/` is storing data (refresh to see)
%tensorboard --logdir /content/neps_colab_playground/results_hpo

In [ ]:
pipeline_space = dict(
    learning_rate=neps.Float(1e-6, 1e-1, log=True),
    optimizer=neps.Categorical(["sgd", "adamw"]),
    num_neurons=neps.Integer(64, 1024),
)

In [ ]:
set_seeds(1)
neps.run(
    evaluate_pipeline=evaluate_pipeline,
    root_directory="results_hpo",
    pipeline_space=pipeline_space,
    evaluations_to_spend=8,
)

In [ ]:
!python -m neps.status results_hpo/

## Key Takeaways
1. **NePS Pattern**: Define function → Define space → Call `neps.run()`
2. **Search Space Types**: `Float`, `Integer`, `Categorical` parameters
3. **Optimization Output**: Tracked in `root_directory` with easy analysis tools
4. **Scalability**: Same API works for complex deep learning tasks

Next steps:
- Learn about **Multi-Fidelity Optimization**